terminal updated version: increase flash size and add IMU

In [ ]:
%%bash
git clone https://github.com/lvgl-micropython/lvgl_micropython.git
cd lvgl_micropython
apt-get update
apt-get install -y gcc g++ make automake python3 git gcc-arm-none-eabi libusb-1.0-0 python3.12-venv cmake python3-pip python3-full
python3 make.py esp32 clean \
  --flash-size=8 \
  BOARD=ESP32_GENERIC_C6 \
  DISPLAY=jd9853 \
  INDEV=axs5106 \
  IMU=qmi8658c

Python version with auto download:

In [ ]:
import subprocess, os, sys, traceback, datetime
from google.colab import files

REPO_DIR = "/content/lvgl_micropython"
BOARD = "ESP32_GENERIC_C6"
FLASH_SIZE = 8
LOG_PATH = "/content/build_log.txt"
ERROR_REPORT_PATH = "/content/build_error_report.txt"

log_lines = []

def log(msg):
    """Print to screen and record to log list simultaneously"""
    print(msg)
    log_lines.append(msg)

def run(cmd, cwd=None, step_name=""):
    """Execute command, print + record output in real-time, raise exception on failure"""
    log(f"\n{'='*60}\n[{step_name}] >>> {cmd}\n{'='*60}")
    process = subprocess.Popen(
        cmd, shell=True, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in process.stdout:
        line = line.rstrip("\n")
        print(line)
        log_lines.append(line)
    process.wait()
    if process.returncode != 0:
        raise RuntimeError(f"Step [{step_name}] failed, exit code {process.returncode}\nCommand: {cmd}")

def save_log():
    with open(LOG_PATH, "w") as f:
        f.write("\n".join(log_lines))

def write_error_report(step_name, error):
    with open(ERROR_REPORT_PATH, "w") as f:
        f.write(f"Build Failure Report\n")
        f.write(f"Time: {datetime.datetime.now()}\n")
        f.write(f"Failed Step: {step_name}\n")
        f.write(f"Error: {error}\n")
        f.write(f"\n{'='*60}\nFull Log (last 200 lines):\n{'='*60}\n")
        f.write("\n".join(log_lines[-200:]))
    files.download(ERROR_REPORT_PATH)

# ---------------------------------------------------------

current_step = "Initialization"
try:
    current_step = "Cloning repository"
    if not os.path.exists(REPO_DIR):
        run(f"git clone https://github.com/lvgl-micropython/lvgl_micropython.git {REPO_DIR}", step_name=current_step)
    else:
        log("Repository already exists, skipping clone")

    current_step = "Installing dependencies"
    run("apt-get update", step_name=current_step)
    run("apt-get install -y gcc g++ make automake python3 git gcc-arm-none-eabi "
        "libusb-1.0-0 python3.12-venv cmake python3-pip python3-full", step_name=current_step)

    current_step = "Building firmware"
    run(
        f"python3 make.py esp32 clean "
        f"--flash-size={FLASH_SIZE} "
        f"BOARD={BOARD} "
        f"DISPLAY=jd9853 "
        f"INDEV=axs5106 "
        f"IMU=qmi8658c",
        cwd=REPO_DIR, step_name=current_step
    )

    current_step = "Locating firmware file"
    bin_path = f"{REPO_DIR}/build/lvgl_micropy_{BOARD}-{FLASH_SIZE}.bin"
    if not os.path.exists(bin_path):
        build_dir = f"{REPO_DIR}/build"
        candidates = [os.path.join(build_dir, f) for f in os.listdir(build_dir) if f.endswith(".bin")]
        if not candidates:
            raise FileNotFoundError("No .bin files found in build directory")
        bin_path = max(candidates, key=os.path.getmtime)

    log(f"\n✅ Build successful: {bin_path}")
    save_log()
    files.download(bin_path)
    files.download(LOG_PATH)  # Also keep a complete log for reference

except Exception as e:
    log(f"\n❌ Failed at step [{current_step}]: {e}")
    log(traceback.format_exc())
    save_log()
    write_error_report(current_step, e)
    print(f"\nError report generated: {ERROR_REPORT_PATH}")

REF:https://peter.quantr.hk/2025/07/waveshare-esp32-c6-1-47-touch-micropython-lvgl/